# 🔍 Becoming a Pattern Detective

**Duration:** 60 minutes  
**Level:** Intermediate

---

## Welcome, Pattern Detective!

Have you ever noticed that your heart beats differently when you're relaxed versus when you're excited? Or that your breathing changes when you're stressed? Your body creates **patterns** in its signals, and learning to recognize these patterns is like becoming a detective for your health!

In this notebook, you'll learn to:
- 🎯 Extract important features from biosignals (like finding clues)
- 📊 Compare patterns between different states (rest vs. active)
- 🤖 Build a simple stress detector
- 🎨 Visualize what you discover

Let's start your detective training!

## 📦 Step 1: Getting Our Detective Tools

First, let's import the tools we'll need for our pattern investigation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

# Make our plots look nice!
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Detective tools loaded and ready!")

## 🫀 Step 2: Creating Our Mystery Signals

Let's create two types of heart rate signals:
- **Relaxed state**: Slow, steady heartbeat (like reading a book)
- **Active state**: Faster, more variable heartbeat (like exercising)

These are our "mystery" signals that we need to learn to tell apart!

In [ ]:
def generate_heartbeat_signal(duration=60, sampling_rate=100, base_hr=70, variability=5, stress_level=0):
    """
    Generate a synthetic heartbeat signal.
    
    Parameters:
    - duration: how long the signal is (in seconds)
    - sampling_rate: how many measurements per second
    - base_hr: average heart rate (beats per minute)
    - variability: how much the heart rate changes
    - stress_level: how stressed (0=calm, 1=very stressed)
    """
    t = np.linspace(0, duration, duration * sampling_rate)
    
    # Create heart rate that varies over time
    hr = base_hr + variability * np.sin(2 * np.pi * 0.1 * t)  # Slow breathing rhythm
    
    # Add random variations (HRV - Heart Rate Variability)
    hrv_noise = np.random.normal(0, variability * (1 + stress_level), len(t))
    hr = hr + hrv_noise
    
    # Add high-frequency stress component
    if stress_level > 0:
        stress_component = stress_level * 10 * np.sin(2 * np.pi * 0.5 * t)
        hr = hr + stress_component
    
    # Create the actual heartbeat signal (peaks represent heartbeats)
    signal_data = np.zeros_like(t)
    current_time = 0
    idx = 0
    
    while idx < len(t):
        # Calculate time to next beat based on current heart rate
        beats_per_second = hr[idx] / 60
        time_to_next_beat = 1 / beats_per_second
        
        # Add a peak (heartbeat)
        peak_width = int(0.15 * sampling_rate)  # Peak lasts ~0.15 seconds
        peak_start = idx
        peak_end = min(idx + peak_width, len(signal_data))
        
        # Create a peak shape
        peak_shape = signal.windows.gaussian(peak_width, std=peak_width/6)
        signal_data[peak_start:peak_end] = peak_shape[:peak_end-peak_start]
        
        # Move to next beat
        idx += int(time_to_next_beat * sampling_rate)
    
    return t, signal_data, hr

# Generate our two mystery signals!
print("🧪 Generating synthetic biosignals...")

# Relaxed signal: lower heart rate, less variability
t_relaxed, signal_relaxed, hr_relaxed = generate_heartbeat_signal(
    duration=60, base_hr=65, variability=3, stress_level=0
)

# Active/stressed signal: higher heart rate, more variability
t_active, signal_active, hr_active = generate_heartbeat_signal(
    duration=60, base_hr=85, variability=8, stress_level=0.7
)

print(f"✅ Created {len(signal_relaxed)} data points for each state!")
print(f"   Sampling rate: 100 Hz (100 measurements per second)")

Let's see what our mystery signals look like!

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Plot relaxed state
axes[0].plot(t_relaxed[:1000], signal_relaxed[:1000], color='blue', linewidth=1.5)
axes[0].set_title('😌 Relaxed State (First 10 seconds)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Signal Amplitude')
axes[0].grid(True, alpha=0.3)

# Plot active state
axes[1].plot(t_active[:1000], signal_active[:1000], color='red', linewidth=1.5)
axes[1].set_title('⚡ Active/Stressed State (First 10 seconds)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Signal Amplitude')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("👀 Can you see the differences? The active state has more peaks (faster heartbeat)!")

## 🔎 Step 3: Extracting Features (Finding Clues)

Now comes the detective work! Instead of looking at thousands of data points, we'll extract **features** - important characteristics that describe the signal.

Think of features like this:
- If signals were people, features would be things like height, weight, hair color
- For biosignals, features are things like average value, how much it varies, number of peaks

Let's create a function that extracts features from any signal!

In [ ]:
def extract_features(signal_data, sampling_rate=100):
    """
    Extract important features from a biosignal.
    Returns a dictionary of features (clues about the signal).
    """
    features = {}
    
    # 1. STATISTICAL FEATURES (Basic math about the signal)
    features['mean'] = np.mean(signal_data)  # Average value
    features['std'] = np.std(signal_data)    # How much it varies (standard deviation)
    features['max'] = np.max(signal_data)    # Highest point
    features['min'] = np.min(signal_data)    # Lowest point
    features['range'] = features['max'] - features['min']  # Total spread
    
    # 2. PEAK FEATURES (Counting heartbeats)
    # Find peaks (heartbeats) - they're points higher than their neighbors
    peaks, _ = signal.find_peaks(signal_data, height=0.3, distance=50)
    features['num_peaks'] = len(peaks)  # Total number of beats
    features['avg_peak_height'] = np.mean(signal_data[peaks]) if len(peaks) > 0 else 0
    
    # Calculate heart rate from peaks
    if len(peaks) > 1:
        # Time between peaks (inter-beat intervals)
        peak_intervals = np.diff(peaks) / sampling_rate  # Convert to seconds
        # Heart rate = 60 seconds / time between beats
        heart_rates = 60 / peak_intervals
        features['avg_heart_rate'] = np.mean(heart_rates)
        features['heart_rate_variability'] = np.std(heart_rates)
    else:
        features['avg_heart_rate'] = 0
        features['heart_rate_variability'] = 0
    
    # 3. FREQUENCY FEATURES (How fast things change)
    # Use FFT (Fast Fourier Transform) to see which frequencies are present
    fft_values = np.fft.fft(signal_data)
    fft_freq = np.fft.fftfreq(len(signal_data), 1/sampling_rate)
    
    # Look at positive frequencies only
    positive_freq_idx = fft_freq > 0
    power = np.abs(fft_values[positive_freq_idx])**2
    freqs = fft_freq[positive_freq_idx]
    
    # Find dominant frequency (which rhythm is strongest)
    dominant_freq_idx = np.argmax(power)
    features['dominant_frequency'] = freqs[dominant_freq_idx]
    features['spectral_energy'] = np.sum(power)
    
    # 4. SHAPE FEATURES (How the signal looks)
    features['skewness'] = skew(signal_data)  # Is it lopsided?
    features['kurtosis'] = kurtosis(signal_data)  # Does it have sharp peaks?
    
    # 5. VARIABILITY FEATURES
    # How much the signal changes from one point to the next
    differences = np.diff(signal_data)
    features['mean_diff'] = np.mean(np.abs(differences))
    features['std_diff'] = np.std(differences)
    
    return features

print("🔧 Feature extraction function ready!")
print("   This is like having a toolkit to measure everything about our signals!")

Now let's extract features from both signals and compare them!

In [ ]:
# Extract features from both signals
features_relaxed = extract_features(signal_relaxed)
features_active = extract_features(signal_active)

# Display them side by side
print("🔍 FEATURE COMPARISON\n" + "="*60)
print(f"{'Feature':<30} {'Relaxed':>12} {'Active':>12}")
print("="*60)

for feature_name in features_relaxed.keys():
    relaxed_val = features_relaxed[feature_name]
    active_val = features_active[feature_name]
    print(f"{feature_name:<30} {relaxed_val:>12.2f} {active_val:>12.2f}")

print("\n💡 Notice the differences!")
print("   - Active state has more peaks (higher heart rate)")
print("   - Active state has higher variability")
print("   - These differences are the 'clues' we use for detection!")

## 📊 Step 4: Visualizing Feature Distributions

Let's generate multiple samples of each state and see how their features compare!
This is like collecting evidence from many cases to find patterns.

In [ ]:
print("🧪 Generating 30 samples of each state...")

# Collect features from multiple samples
n_samples = 30
relaxed_features_list = []
active_features_list = []

for i in range(n_samples):
    # Generate relaxed signal
    _, sig_rel, _ = generate_heartbeat_signal(
        duration=30, base_hr=65 + np.random.normal(0, 3), 
        variability=3 + np.random.normal(0, 0.5), stress_level=0
    )
    relaxed_features_list.append(extract_features(sig_rel))
    
    # Generate active signal
    _, sig_act, _ = generate_heartbeat_signal(
        duration=30, base_hr=85 + np.random.normal(0, 5), 
        variability=8 + np.random.normal(0, 1), stress_level=0.7
    )
    active_features_list.append(extract_features(sig_act))

print(f"✅ Collected features from {n_samples} samples of each state!")

# Extract specific features for visualization
relaxed_hr = [f['avg_heart_rate'] for f in relaxed_features_list]
active_hr = [f['avg_heart_rate'] for f in active_features_list]

relaxed_hrv = [f['heart_rate_variability'] for f in relaxed_features_list]
active_hrv = [f['heart_rate_variability'] for f in active_features_list]

relaxed_peaks = [f['num_peaks'] for f in relaxed_features_list]
active_peaks = [f['num_peaks'] for f in active_features_list]

In [ ]:
# Create beautiful comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Heart Rate comparison
axes[0, 0].hist(relaxed_hr, alpha=0.6, label='Relaxed', color='blue', bins=15)
axes[0, 0].hist(active_hr, alpha=0.6, label='Active', color='red', bins=15)
axes[0, 0].set_xlabel('Average Heart Rate (bpm)', fontsize=11)
axes[0, 0].set_ylabel('Count', fontsize=11)
axes[0, 0].set_title('🫀 Heart Rate Distribution', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: HRV comparison
axes[0, 1].hist(relaxed_hrv, alpha=0.6, label='Relaxed', color='blue', bins=15)
axes[0, 1].hist(active_hrv, alpha=0.6, label='Active', color='red', bins=15)
axes[0, 1].set_xlabel('Heart Rate Variability', fontsize=11)
axes[0, 1].set_ylabel('Count', fontsize=11)
axes[0, 1].set_title('📊 Variability Distribution', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Peak count comparison
axes[1, 0].hist(relaxed_peaks, alpha=0.6, label='Relaxed', color='blue', bins=15)
axes[1, 0].hist(active_peaks, alpha=0.6, label='Active', color='red', bins=15)
axes[1, 0].set_xlabel('Number of Peaks (beats)', fontsize=11)
axes[1, 0].set_ylabel('Count', fontsize=11)
axes[1, 0].set_title('🔺 Peak Count Distribution', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Scatter plot - HR vs HRV
axes[1, 1].scatter(relaxed_hr, relaxed_hrv, alpha=0.6, label='Relaxed', color='blue', s=100)
axes[1, 1].scatter(active_hr, active_hrv, alpha=0.6, label='Active', color='red', s=100)
axes[1, 1].set_xlabel('Average Heart Rate (bpm)', fontsize=11)
axes[1, 1].set_ylabel('Heart Rate Variability', fontsize=11)
axes[1, 1].set_title('🎯 HR vs HRV (2D View)', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🎨 Beautiful! You can clearly see the two groups separate!")
print("   This separation is what makes classification possible!")

## 🤖 Step 5: Building a Simple Stress Detector

Now for the exciting part! Let's build a **threshold-based classifier**.

**How it works:**
- If the average heart rate is above a certain threshold, we predict "Active/Stressed"
- If it's below the threshold, we predict "Relaxed"

It's like saying: "If someone's heart rate is over 75 bpm, they're probably active!"

In [ ]:
class SimpleStressDetector:
    """
    A simple detector that classifies states based on heart rate threshold.
    """
    def __init__(self):
        self.threshold = None
        
    def train(self, relaxed_hrs, active_hrs):
        """
        Find the best threshold by looking at the data.
        """
        # A simple approach: use the midpoint between the averages
        avg_relaxed = np.mean(relaxed_hrs)
        avg_active = np.mean(active_hrs)
        self.threshold = (avg_relaxed + avg_active) / 2
        
        print(f"🎓 Training complete!")
        print(f"   Average HR (Relaxed): {avg_relaxed:.1f} bpm")
        print(f"   Average HR (Active): {avg_active:.1f} bpm")
        print(f"   Decision threshold: {self.threshold:.1f} bpm")
        
    def predict(self, heart_rate):
        """
        Predict the state based on heart rate.
        """
        if heart_rate > self.threshold:
            return "Active/Stressed"
        else:
            return "Relaxed"
    
    def evaluate(self, test_relaxed_hrs, test_active_hrs):
        """
        Test the detector and calculate accuracy.
        """
        correct = 0
        total = len(test_relaxed_hrs) + len(test_active_hrs)
        
        # Test on relaxed samples
        for hr in test_relaxed_hrs:
            if self.predict(hr) == "Relaxed":
                correct += 1
        
        # Test on active samples
        for hr in test_active_hrs:
            if self.predict(hr) == "Active/Stressed":
                correct += 1
        
        accuracy = 100 * correct / total
        return accuracy

# Create and train our detector
detector = SimpleStressDetector()
detector.train(relaxed_hr, active_hr)

Let's test our detector!

In [ ]:
# Generate new test data (data the detector hasn't seen before)
print("🧪 Generating test data...\n")

test_relaxed_hr = []
test_active_hr = []

for i in range(20):
    # Relaxed test samples
    _, sig, _ = generate_heartbeat_signal(
        duration=30, base_hr=65 + np.random.normal(0, 3), 
        variability=3, stress_level=0
    )
    features = extract_features(sig)
    test_relaxed_hr.append(features['avg_heart_rate'])
    
    # Active test samples
    _, sig, _ = generate_heartbeat_signal(
        duration=30, base_hr=85 + np.random.normal(0, 5), 
        variability=8, stress_level=0.7
    )
    features = extract_features(sig)
    test_active_hr.append(features['avg_heart_rate'])

# Evaluate the detector
accuracy = detector.evaluate(test_relaxed_hr, test_active_hr)

print(f"🎯 RESULTS")
print(f"   Tested on: {len(test_relaxed_hr) + len(test_active_hr)} samples")
print(f"   Accuracy: {accuracy:.1f}%")

if accuracy > 90:
    print("\n🎉 Excellent! Your detector is working great!")
elif accuracy > 75:
    print("\n👍 Good job! There's room for improvement, but this is solid!")
else:
    print("\n🤔 Hmm, we might need to improve our approach!")

# Test on a few individual cases
print("\n🔬 Testing individual predictions:")
for i in range(5):
    hr = test_relaxed_hr[i]
    pred = detector.predict(hr)
    print(f"   HR: {hr:.1f} bpm → Predicted: {pred}")

## 📈 Step 6: Visualizing the Detector's Decision

Let's visualize how our detector makes decisions!

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Plot all the test samples
ax.scatter(test_relaxed_hr, [0]*len(test_relaxed_hr), 
           c='blue', s=100, alpha=0.6, label='Relaxed (True)', marker='o')
ax.scatter(test_active_hr, [1]*len(test_active_hr), 
           c='red', s=100, alpha=0.6, label='Active (True)', marker='^')

# Draw the decision threshold
ax.axvline(detector.threshold, color='green', linestyle='--', linewidth=3,
           label=f'Decision Threshold ({detector.threshold:.1f} bpm)')

# Shade the decision regions
ax.axvspan(0, detector.threshold, alpha=0.2, color='blue', label='Predicts: Relaxed')
ax.axvspan(detector.threshold, 120, alpha=0.2, color='red', label='Predicts: Active')

ax.set_xlabel('Heart Rate (bpm)', fontsize=13)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Relaxed', 'Active'])
ax.set_title('🎯 Stress Detector Decision Boundary', fontsize=15, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 What you're seeing:")
print("   - Points left of the green line are classified as 'Relaxed'")
print("   - Points right of the green line are classified as 'Active'")
print("   - Any blue circles on the right or red triangles on the left are mistakes!")

## 🎓 Exercises & Challenges

Now it's your turn to become a master pattern detective!

### Exercise 1: Create Your Own Feature
Can you think of a new feature that might help distinguish between relaxed and active states?
Try modifying the `extract_features` function to add it!

In [ ]:
# YOUR CODE HERE
# Try adding a new feature to the extract_features function
# Ideas:
#   - Energy in a specific frequency band
#   - Ratio between high and low frequencies
#   - Maximum rate of change in the signal
#   - Zero crossing rate (how often the signal crosses zero)

print("💪 Try it yourself! Add your feature above.")

### Exercise 2: Improve the Detector
Can you improve the detector by using multiple features instead of just heart rate?
Maybe use both heart rate AND heart rate variability!

In [ ]:
class AdvancedStressDetector:
    """
    YOUR TURN: Build a better detector using multiple features!
    """
    def __init__(self):
        # Add your thresholds or decision rules here
        pass
    
    def train(self, relaxed_features, active_features):
        # YOUR CODE: Learn from the training data
        pass
    
    def predict(self, features):
        # YOUR CODE: Make a prediction
        pass

print("🚀 Challenge accepted! Try building your improved detector above.")
print("   Hint: You could use multiple thresholds or combine features!")

### Exercise 3: Test Different Thresholds
Try different threshold values and see how they affect accuracy!

In [ ]:
# Test different thresholds
thresholds_to_test = [70, 72, 75, 78, 80]
accuracies = []

for threshold in thresholds_to_test:
    # Create a detector with this threshold
    test_detector = SimpleStressDetector()
    test_detector.threshold = threshold
    
    # Evaluate it
    accuracy = test_detector.evaluate(test_relaxed_hr, test_active_hr)
    accuracies.append(accuracy)
    print(f"Threshold: {threshold} bpm → Accuracy: {accuracy:.1f}%")

# Find the best threshold
best_idx = np.argmax(accuracies)
print(f"\n🏆 Best threshold: {thresholds_to_test[best_idx]} bpm with {accuracies[best_idx]:.1f}% accuracy!")

## 🎉 Congratulations, Pattern Detective!

You've learned so much in this notebook:

✅ How to generate and visualize biosignals  
✅ How to extract meaningful features from raw data  
✅ How to compare patterns between different states  
✅ How to build a simple classifier using thresholds  
✅ How to evaluate and visualize your detector's performance  

### 🚀 What's Next?

In the next notebook, we'll take this to the next level by using **machine learning**! Instead of manually choosing thresholds, we'll teach a computer to automatically learn the best decision rules.

### 💭 Think About This:

- What other features could you create from biosignals?
- How would you handle signals where the person is somewhere between relaxed and stressed?
- Could you use these techniques for other biosignals like brain waves or muscle activity?

### 📚 Key Concepts You Mastered:

- **Feature Extraction**: Converting raw data into meaningful measurements
- **Pattern Recognition**: Finding differences between groups
- **Threshold Classification**: Making decisions based on simple rules
- **Evaluation**: Measuring how well your detector works

---

**Keep exploring, keep learning, and remember:** Every great AI system starts with understanding patterns in data. You're on your way to becoming a biosignal expert! 🌟